# 03 · Baselines: JET-only, random, Isolation Forest, LOF
Step 3 of the thesis workflow (after `02_encoding_onehot`).

**Input:** `data/derived/meta.parquet` (notebook 01), `data/derived/X_onehot.npz` + `onehot_features.json` (notebook 02)
**Output:** rows in `results/runs.csv` (one per method × seed × budget, with the git commit)

| Baseline | Score (higher = more suspicious) | List |
|---|---|---|
| Random, whole ledger | 0 for all (pure random order) | all entries — lower bound |
| Random inside JET union | 0 inside the union | JET entries only — lower bound for the prefilter |
| JET-only | h = number of JETs hit, random ties | JET entries only |
| Isolation Forest | −score_samples | all entries (model-only) |
| LOF | −negative_outlier_factor_ | all entries (model-only) |

Rules: models fitted per fiscal year without labels (D6); hyperparameters chosen by mean AP on FY1 = 2023 only (D15); results reported on FY2 = 2024; metrics and tie-break as in D14 (`src/metrics.py`); main feature set v2 (D4, D11, D13: 132 columns); budgets k = 25 … 2,000 as in Gronewald, Scherer et al. (2026), main k = 100 and 1,000 (D14).

In [1]:
# cell_03_01
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
RAW_REL = Path("data/raw/journal_entries_4203.csv")


def find_project_root(marker: Path = RAW_REL) -> Path:
    # walk up from the working directory until `marker` exists (same as notebooks 01, 02)
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / marker).exists():
            return p
    raise FileNotFoundError(f"{marker} not found from {here} upwards")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
from src import encode as fe, metrics as m, detectors as det

DERIVED = ROOT / "data" / "derived"
RUNS = ROOT / "results" / "runs.csv"
FY1, FY2 = 2023, 2024
GROUPS = fe.FEATURE_GROUPS_MAIN
print("project root:", ROOT.name, "| commit:", m.git_commit(ROOT))

project root: Jet | commit: 61bb565+dirty


## 1. Load meta and the encoding (main feature set), split by fiscal year

In [2]:
# cell_03_02
meta = pd.read_parquet(DERIVED / "meta.parquet")
X_all, feature_info, params = fe.load_encoding(DERIVED / "X_onehot.npz", DERIVED / "onehot_features.json")
X_main, names = fe.select_columns(X_all, feature_info, GROUPS)

data = {}
for fy in (FY1, FY2):
    idx = np.flatnonzero(meta["fy"].to_numpy() == fy)
    d = meta.iloc[idx].reset_index(drop=True)
    data[fy] = {"meta": d, "X": X_main[idx].toarray().astype(np.float32),
                "y": d["label"].to_numpy(), "budgets": m.budgets(len(d))}
    print(f"FY{fy}: {len(d):,} entries, {d['label'].sum()} anomalies, X {data[fy]['X'].shape}, "
          f"budgets {data[fy]['budgets']}")
print("feature groups:", fe.groups_label(GROUPS))

FY2023: 51,064 entries, 423 anomalies, X (51064, 132), budgets {'k25': 25, 'k50': 50, 'k100': 100, 'k250': 250, 'k500': 500, 'k1000': 1000, 'k2000': 2000}
FY2024: 51,365 entries, 427 anomalies, X (51365, 132), budgets {'k25': 25, 'k50': 50, 'k100': 100, 'k250': 250, 'k500': 500, 'k1000': 1000, 'k2000': 2000}
feature groups: lines+calendar_flags+lag_days@v2


## 2. Duplicates in the encoding — why the LOF grid starts at k = 50

Many entries are identical after encoding (same accounts, user, amounts, calendar flags: amounts are quantity × fixed price, salaries repeat every month). For LOF, a point with k or more identical neighbours has k-distance 0, which makes the density estimate unstable (Campos et al. 2016, "Duplicates"). The LOF grid therefore only uses k larger than the largest group of identical entries.

In [3]:
# cell_03_03
for fy in (FY1, FY2):
    u, counts = np.unique(data[fy]["X"], axis=0, return_counts=True)
    print(f"FY{fy}: {len(u):,} unique vectors of {len(data[fy]['X']):,}; "
          f"{counts[counts > 1].sum():,} entries have an identical twin; largest group = {counts.max()}")
print("LOF grid:", [p["n_neighbors"] for p in det.GRIDS["lof"]])

FY2023: 24,207 unique vectors of 51,064; 36,137 entries have an identical twin; largest group = 36
FY2024: 24,278 unique vectors of 51,365; 36,472 entries have an identical twin; largest group = 36
LOF grid: [50, 100, 200]


## 3. Rule baselines and random lower bounds (both years)

In [4]:
# cell_03_04
rule_scores = {
    "random_ledger": lambda d: np.zeros(len(d)),
    "random_jet_union": lambda d: np.where(d["jet_any"] == 1, 0.0, np.nan),
    "jet_only": lambda d: np.where(d["jet_any"] == 1, d["h"].astype(float), np.nan),
}
results = {}
for fy in (FY1, FY2):
    for name, f in rule_scores.items():
        per_seed = m.evaluate(f(data[fy]["meta"]), data[fy]["y"], budget_map=data[fy]["budgets"])
        results[(name, fy)] = per_seed
        m.append_runs(per_seed, RUNS, method=name, fy=fy, notes="rule / random baseline", repo_root=ROOT)
pd.concat({f"{n} FY{fy}": m.summarise(r).set_index("budget")[["precision_mean", "recall_mean", "ap_mean"]]
           for (n, fy), r in results.items() if fy == FY2}, axis=1).round(3)

random_ledger FY2024                     random_jet_union FY2024                     jet_only FY2024                    
             precision_mean recall_mean ap_mean          precision_mean recall_mean ap_mean  precision_mean recall_mean ap_mean
budget                                                                                                                         
k25                   0.016       0.001   0.009                   0.016       0.001    0.03           0.536       0.031   0.092
k50                   0.016       0.002   0.009                   0.044       0.005    0.03           0.480       0.056   0.092
k100                  0.012       0.003   0.009                   0.038       0.009    0.03           0.300       0.070   0.092
k250                  0.013       0.007   0.009                   0.030       0.017    0.03           0.162       0.095   0.092
k500                  0.011       0.013   0.009                   0.030       0.035    0.03           0.112       0.131   0.092
k1000                 0.009       0.022   0.009                   0.033       0.076    0.03           0.094       0.220   0.092
k2000                 0.009       0.043   0.009                   0.030       0.140    0.03           0.070       0.329   0.092

## 4. Tune Isolation Forest and LOF on FY1 (2023) — mean AP over 5 seeds

Only FY1 labels are used here. The setting with the highest mean AP is then used, unchanged, on FY2.

In [5]:
# cell_03_05
tuning = []
for detector, grid in det.GRIDS.items():
    for p in grid:
        per_seed, info = det.run(detector, data[FY1]["X"], data[FY1]["y"], p, budget_map=data[FY1]["budgets"])
        m.append_runs(per_seed, RUNS, method="model_only", detector=detector, encoding="onehot",
                      feature_groups=fe.groups_label(GROUPS), fy=FY1,
                      notes="tuning " + det.params_label(p), repo_root=ROOT)
        s = m.summarise(per_seed).set_index("budget")
        tuning.append({"detector": detector, "params": det.params_label(p), "ap_mean": s["ap_mean"].iloc[0],
                       "P@100": s.loc["k100", "precision_mean"], "P@1000": s.loc["k1000", "precision_mean"],
                       "R@1000": s.loc["k1000", "recall_mean"], "seconds": info["seconds"]})
tuning = pd.DataFrame(tuning)
tuning.round(4)

,detector,params,ap_mean,P@100,P@1000,R@1000,seconds
0,iforest,n_estimators=100;max_samples=256,0.0476,0.038,0.0754,0.1783,2.6
1,iforest,n_estimators=300;max_samples=256,0.0461,0.030,0.0768,0.1816,7.2
2,iforest,n_estimators=300;max_samples=2048,0.0557,0.012,0.0860,0.2033,9.2
3,lof,n_neighbors=50,0.1161,0.270,0.1960,0.4634,5.7
4,lof,n_neighbors=100,0.1008,0.160,0.1600,0.3783,5.6
5,lof,n_neighbors=200,0.0841,0.030,0.1120,0.2648,5.9


In [6]:
# cell_03_06
best = {}
for detector, grid in det.GRIDS.items():
    t = tuning[tuning["detector"] == detector].reset_index(drop=True)
    i = int(t["ap_mean"].idxmax())
    best[detector] = grid[i]
    print(f"{detector}: best on FY1 = {det.params_label(grid[i])} (AP {t.loc[i, 'ap_mean']:.4f})")

# D15 edge rule: if the best setting lies on the edge of its grid, evaluate one step further out on FY1
for detector, p in list(best.items()):
    ext = det.edge_extension(detector, p, data[FY1]["X"])
    if ext is None:
        print(f"{detector}: best setting not on the grid edge")
        continue
    per_seed, info = det.run(detector, data[FY1]["X"], data[FY1]["y"], ext, budget_map=data[FY1]["budgets"])
    m.append_runs(per_seed, RUNS, method="model_only", detector=detector, encoding="onehot",
                  feature_groups=fe.groups_label(GROUPS), fy=FY1,
                  notes="tuning (edge rule) " + det.params_label(ext), repo_root=ROOT)
    ap_ext = m.summarise(per_seed)["ap_mean"].iloc[0]
    ap_best = tuning.loc[(tuning["detector"] == detector) & (tuning["params"] == det.params_label(p)), "ap_mean"].iloc[0]
    print(f"{detector}: edge step {det.params_label(ext)} AP {ap_ext:.4f} vs {ap_best:.4f}")
    if ap_ext > ap_best:
        best[detector] = ext

iforest: best on FY1 = n_estimators=300;max_samples=2048 (AP 0.0557)
lof: best on FY1 = n_neighbors=50 (AP 0.1161)
iforest: edge step n_estimators=300;max_samples=4096 AP 0.0637 vs 0.0557
lof: edge step n_neighbors=40 AP 0.1179 vs 0.1161


## 5. Report on FY2 (2024) with the FY1 settings

In [7]:
# cell_03_07
model_scores = {}
for detector, p in best.items():
    per_seed, info = det.run(detector, data[FY2]["X"], data[FY2]["y"], p, budget_map=data[FY2]["budgets"])
    results[(detector, FY2)] = per_seed
    model_scores[detector] = info["scores"]
    m.append_runs(per_seed, RUNS, method="model_only", detector=detector, encoding="onehot",
                  feature_groups=fe.groups_label(GROUPS), fy=FY2,
                  notes="report " + det.params_label(p), repo_root=ROOT)
    print(detector, det.params_label(p), f"{info['seconds']} s")

iforest n_estimators=300;max_samples=4096 9.8 s
lof n_neighbors=40 7.6 s


## 6. FY2024 comparison (mean over 5 seeds)

In [8]:
# cell_03_08
order = ["random_ledger", "random_jet_union", "jet_only", "iforest", "lof"]
rows = []
for name in order:
    s = m.summarise(results[(name, FY2)]).set_index("budget")
    rows.append({"method": name, "AP": s["ap_mean"].iloc[0],
                 **{f"P@{b[1:]}": s.loc[b, "precision_mean"] for b in m.budgets()},
                 "R@1000": s.loc["k1000", "recall_mean"], "FP@1000": s.loc["k1000", "fp_mean"]})
comparison = pd.DataFrame(rows).set_index("method")
comparison.round(3)

,AP,P@25,P@50,P@100,P@250,P@500,P@1000,P@2000,R@1000,FP@1000
method,,,,,,,,,,
random_ledger,0.009,0.016,0.016,0.012,0.013,0.011,0.009,0.009,0.022,990.8
random_jet_union,0.030,0.016,0.044,0.038,0.030,0.030,0.033,0.030,0.076,967.4
jet_only,0.092,0.536,0.480,0.300,0.162,0.112,0.094,0.070,0.220,906.0
iforest,0.059,0.000,0.000,0.038,0.104,0.073,0.085,0.071,0.198,915.4
lof,0.144,0.320,0.400,0.310,0.284,0.240,0.191,0.104,0.447,809.0


## 7. Checks

- **Single 4-line entry `p_2`** (FY2024, normal): it alone fills slot 4. Where does it rank?
- **Anomalies inside vs outside the JET union** found by the models within k = 1,000 (RQ3 preview).

In [9]:
# cell_03_09
d2 = data[FY2]["meta"]
i_p2 = int(np.flatnonzero(d2["posting_id"].to_numpy() == "p_2")[0])
for detector, s in model_scores.items():
    rank = int((s > s[i_p2]).sum()) + 1
    print(f"{detector}: p_2 ranks {rank:,} of {len(s):,}")

iforest: p_2 ranks 1,357 of 51,365
lof: p_2 ranks 209 of 51,365


In [10]:
# cell_03_10
k5 = data[FY2]["budgets"]["k1000"]
y2, jet2 = data[FY2]["y"].astype(bool), d2["jet_any"].to_numpy() == 1
for detector, s in model_scores.items():
    top = m.review_order(s, 0)[:k5]
    found = y2[top]
    print(f"{detector}: top {k5} holds {found.sum()} anomalies — {(found & jet2[top]).sum()} inside the JET union, "
          f"{(found & ~jet2[top]).sum()} outside (of {(y2 & ~jet2).sum()} outside in total)")

iforest: top 1000 holds 88 anomalies — 87 inside the JET union, 1 outside (of 31 outside in total)
lof: top 1000 holds 191 anomalies — 168 inside the JET union, 23 outside (of 31 outside in total)


## Notes for the thesis

- Ch. 3 (detectors, hyperparameter selection): grids and the FY1-only selection rule (D15); the duplicate check that sets the LOF grid.
- Ch. 4 (baselines): the FY2024 comparison table; JET-only is strong only in its top ~56 entries (h ≥ 3).
- Week 3 adds the autoencoder, H1, H2 and the embedding encoding with the same functions.
- Encoding v2 (25 Nov review): rows in `results/runs.csv` carry `@v2` in `feature_groups`; older rows are v1 (with promptly and tax_rate, FY1-only scalers) and are not compared with v2 rows.